Goal:

implement an algorithm that tells us whether or not two shapes can be merged

In [1]:
def unravel(shape:tuple[int], offset: int) -> list[int]:
  # same as on tinygrad.shape.view.unravel, but with different types
  # find the position of offset on each dimension based on shape
  # similar to unravel_index in numpy/torch
  acc, idxs = 1, []
  for d in reversed(shape):
    idxs.append((offset//acc)%d)
    acc *= d
  return idxs[::-1]

In [18]:
def index_fn(stride: tuple[int], index: tuple[int]) -> int:
  assert len(stride) == len(index)
  return sum(s * i for s, i in zip(stride, index))

In [15]:
shape1 = [10,3,3]
stride1 = [5,2,3]
shape2 = [8]
stride2 = [4]

In [16]:
for k in range(shape2[-1]):
  print(unravel(shape1, k * stride2[-1]))


[0, 0, 0]
[0, 1, 1]
[0, 2, 2]
[1, 1, 0]
[1, 2, 1]
[2, 0, 2]
[2, 2, 0]
[3, 0, 1]


In [9]:
index_fn(shape1, (0, 2, 2))

12

Goal: calculate what overflows are happening

In [13]:
def calculate_overflows_1d(shape: tuple[int], base: int, max_k: int):
  if max_k < 2:
    return []

  result = []
  
  base_t = unravel(shape, base)

  prev = base_t.copy()
  for k in range(2, max_k):
    cur = unravel(shape, k * base)

    carry_ints = []
    for j in range(len(shape)):
      if cur[j] < prev[j] + base_t[j]:
        carry_ints.append(j)

    if carry_ints:
      result.append((k, carry_ints))

    prev = cur
  
  return result


In [17]:
calculate_overflows_1d(shape1, stride2[-1], shape2[-1])

[(3, [1, 2]), (5, [1]), (6, [2]), (7, [1])]

Now, we have the criterion; just also do this for multi-dim'l shapes.

In [19]:
def one_hot_tuple(n: int, j: int) -> tuple[int]:
  assert j < n
  return tuple(1 if i == j else 0 for i in range(n))

In [50]:
def increment_tuple(t: tuple[int], j: int) -> tuple[int]:
  assert j < len(t)
  return tuple(t[i] if i != j else t[i] + 1 for i in range(len(t)))

In [55]:
from typing import Iterator

def _index_iterator(shape: tuple[int], starting_index: int = 0) -> Iterator[tuple[int]]:
  if (not shape) or starting_index >= len(shape):
    yield tuple()
    return

  for i in range(shape[starting_index]):
    for suffix in _index_iterator(shape, starting_index=starting_index + 1):
      yield (i, ) + suffix

def index_iterator(shape: tuple[int]) -> Iterator[tuple[int]]:
  yield from _index_iterator(shape)


In [39]:
def calculate_carries(base_rep: tuple[int], next_rep: tuple[int], increment_rep: tuple[int]) -> tuple[int]:
  """Returns all the indices that had a carry when adding increment_rep to base_rep, resulting in next_rep"""
  carries = []
  for j in range(len(base_rep)):
    if base_rep[j] + increment_rep[j] > next_rep[j]:
      carries.append(j)
  
  return tuple(carries)

# calculate_carries([3,1,5], [4,1,0], [0, 1, 1])

In [58]:
def calculate_overflows(shape1: tuple[int], shape2: tuple[int], stride2: tuple[int]):
  carries = []
  increment_reps = [unravel(shape1, index_fn(stride2, one_hot_tuple(len(stride2), j))) for j in range(len(stride2))]


  for idx in index_iterator(shape2):
    current_repr = unravel(shape1, index_fn(stride2, idx))

    for j in range(len(stride2)):
      if idx[j] + 1 == shape2[j]:
        continue
        
      next_repr = unravel(shape1, index_fn(stride2, increment_tuple(idx, j)))
      carry = calculate_carries(current_repr, next_repr, increment_reps[j])
      
      if carry:
        carries.append((idx, j, carry))
  
  return carries

In [59]:
calculate_overflows(shape1, shape2, stride2)

[((2,), 0, (1, 2)), ((4,), 0, (1,)), ((5,), 0, (2,)), ((6,), 0, (1,))]

In [60]:
calculate_overflows(shape1, (2, 8), (1, 4))

[((0, 2), 0, (1, 2)),
 ((0, 2), 1, (1, 2)),
 ((0, 4), 1, (1,)),
 ((0, 5), 0, (2,)),
 ((0, 5), 1, (2,)),
 ((0, 6), 1, (1,)),
 ((1, 1), 1, (1, 2)),
 ((1, 4), 1, (1, 2)),
 ((1, 6), 1, (1,))]

Now, we need to check if the equations for the given carries hold.

In [62]:
def check_carry_equation(shape: tuple[int], stride: tuple[int], carry_positions: tuple[int]) -> bool:
  running_sum = 0

  for j in carry_positions:
    running_sum += stride[j - 1] - shape[j] * stride[j]


  return running_sum == 0

In [69]:
check_carry_equation([10, 8, 3], [7 * 7 + 3 * 13,7,13], (1,2 ))

True

In [70]:
def is_mergeable_through_carry(shape1: tuple[int], stride1: tuple[int], shape2: tuple[int], stride2: tuple[int]) -> bool:
  overflows = calculate_overflows(shape1, shape2, stride2)

  for overflow in overflows:
    if not check_carry_equation(shape1, stride1, overflow[2]):
      return False
  return True

is_mergeable_through_carry(shape1, stride1, shape2, stride2)

False

In [83]:
assert is_mergeable_through_carry([10, 3, 3], [2 * 7 + 3 * 11, 7, 11], [8], [4]) == False
assert is_mergeable_through_carry([10, 3, 3], [2 * 7 + 3 * 11, 7, 11], [5], [4]) == True

# Direct check

Instead of first checking for all the carries, and then checking the equations for the queries, we may as well check directly for every index, which is closer to the current statement of the theorem.

In [72]:
def composed_index_fn(shape1: tuple[int], stride1: tuple[int], shape2: tuple[int], stride2: tuple[int], idx: tuple[int]) -> int:
  return index_fn(stride1, unravel(shape1, index_fn(stride2, idx)))


In [73]:
def is_mergeable_direct(shape1: tuple[int], stride1: tuple[int], shape2: tuple[int], stride2: tuple[int]) -> bool:
  merged_strides = [index_fn(stride1, unravel(shape1, index_fn(stride2, one_hot_tuple(len(stride2), j)))) for j in range(len(stride2))]

  for idx in index_iterator(shape2):
    for j in range(len(stride2)):
      if idx[j] + 1 == shape2[j]:
        continue

      if composed_index_fn(shape1, stride1, shape2, stride2, idx) + merged_strides[j] != composed_index_fn(shape1, stride1, shape2, stride2, increment_tuple(idx, j)):
        return False

  return True

is_mergeable_direct(shape1, stride1, shape2, stride2)


False

In [82]:
assert is_mergeable_direct([10, 3, 3], [2 * 7 + 3 * 11, 7, 11], [8], [4]) == False
assert is_mergeable_direct([10, 3, 3], [2 * 7 + 3 * 11, 7, 11], [5], [4]) == True